In [1]:
## required libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("data/train.csv")
df

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


In [3]:
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns")
print(df.info())
print(df.isna().sum())
df.duplicated().any()

The dataset has 594194 rows and 21 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 

np.False_

## Data Cleaning 

In [4]:
if 'id' in df.columns:
    df = df.drop('id', axis=1)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in internet_cols:
    df[col] = df[col].replace({'No internet service': 'No'})

df['MultipleLines'] = df['MultipleLines'].replace({'No phone service': 'No'})

binary_cols = internet_cols + ['MultipleLines', 'Partner', 'Dependents', 
                               'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print("Data Cleaning Complete!")
print("-" * 10)
print("Remaining Categorical Columns (Need One-Hot Encoding later):")
print(df.select_dtypes(include=['object']).columns.tolist())
print("-" * 10)
print("Missing values after cleaning:")
print(df.isna().sum().sum())

Data Cleaning Complete!
----------
Remaining Categorical Columns (Need One-Hot Encoding later):
['InternetService', 'Contract', 'PaymentMethod']
----------
Missing values after cleaning:
0


In [5]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,1,0,1,1,29,1,0,DSL,1,0,1,1,0,0,One year,1,Mailed check,60.10,1653.85,0
1,1,0,1,1,58,1,0,DSL,1,1,0,1,1,0,Two year,0,Credit card (automatic),69.50,3778.20,0
2,1,0,1,0,58,1,1,Fiber optic,0,1,0,0,1,1,Month-to-month,1,Electronic check,100.40,5841.35,0
3,0,0,0,0,1,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,69.70,70.70,1
4,0,0,0,0,1,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.45,70.45,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,1,0,0,0,57,1,1,Fiber optic,0,0,1,0,1,1,Two year,0,Bank transfer (automatic),97.55,5460.70,0
594190,0,0,0,0,72,1,1,DSL,1,1,1,1,1,1,Two year,0,Bank transfer (automatic),91.95,6782.15,0
594191,0,0,1,0,72,1,1,No,0,0,0,0,0,0,Two year,0,Credit card (automatic),24.40,1871.90,0
594192,0,0,0,0,32,1,1,Fiber optic,0,0,0,0,0,1,Month-to-month,1,Electronic check,86.00,2847.20,0


In [6]:
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (475355, 19)
Test set shape: (118839, 19)


In [7]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = ['InternetService', 'Contract', 'PaymentMethod']

preprocessor = ColumnTransformer(
    transformers=[
     
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ],
    remainder='passthrough' 
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Optional: Get column names back if you want to view it as a DataFrame later
# cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
# all_feature_names = num_cols + list(cat_feature_names) + [col for col in X_train.columns if col not in num_cols + cat_cols]

In [8]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Ridge Classifier': RidgeClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0) 
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Dictionary to store the results
baseline_results = {}

print("Evaluating Baseline Models (ROC AUC)...\n" + "-"*40)

for name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring='roc_auc')
    
    baseline_results[name] = cv_scores.mean()
    
    print(f"{name:<20} | Mean AUC: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")

print("\n" + "="*40)
print("="*40)
sorted_results = sorted(baseline_results.items(), key=lambda x: x[1], reverse=True)
for name, score in sorted_results:
    print(f"{name:<20} : {score:.4f}")

Evaluating Baseline Models (ROC AUC)...
----------------------------------------
Logistic Regression  | Mean AUC: 0.9078 | Std: 0.0011
Ridge Classifier     | Mean AUC: 0.8988 | Std: 0.0014
XGBoost              | Mean AUC: 0.9151 | Std: 0.0010
LightGBM             | Mean AUC: 0.9150 | Std: 0.0010
CatBoost             | Mean AUC: 0.9158 | Std: 0.0010

CatBoost             : 0.9158
XGBoost              : 0.9151
LightGBM             : 0.9150
Logistic Regression  : 0.9078
Ridge Classifier     : 0.8988
